# Task 3 - DIM Customer and the partitions

In [1]:
df = spark.read.parquet("/user/student/churn/warehouse/dim_customer")

In [2]:
df.printSchema()

root
 |-- cust_key: long (nullable = true)
 |-- customer_id: string (nullable = true)
 |-- age: integer (nullable = true)
 |-- gender: string (nullable = true)
 |-- tenure_months: integer (nullable = true)
 |-- credit_score: integer (nullable = true)
 |-- num_products: integer (nullable = true)
 |-- is_churned: integer (nullable = true)
 |-- churn_month: string (nullable = true)
 |-- clv_ltv: double (nullable = true)
 |-- avg_monthly_balance: double (nullable = true)
 |-- avg_ticket_res_time_hrs: double (nullable = true)
 |-- total_tickets: integer (nullable = true)
 |-- high_severity_tickets: integer (nullable = true)
 |-- offers_received: integer (nullable = true)
 |-- offers_accepted: integer (nullable = true)
 |-- offer_acceptance_rate: double (nullable = true)
 |-- dw_start_date: date (nullable = true)
 |-- dw_end_date: date (nullable = true)
 |-- is_current: integer (nullable = true)
 |-- geography: string (nullable = true)



In [3]:
print("rows:", df.count())

rows: 10000


In [4]:
df.limit(25).toPandas()

,cust_key,customer_id,age,gender,tenure_months,credit_score,num_products,is_churned,churn_month,clv_ltv,...,avg_ticket_res_time_hrs,total_tickets,high_severity_tickets,offers_received,offers_accepted,offer_acceptance_rate,dw_start_date,dw_end_date,is_current,geography
0,3,15565714,47,Male,12,601,2,0,None,1494.73,...,NaN,2,0,4,0,0.000000,2024-12-31,9999-12-31,1,France
1,6,15565806,38,Male,108,532,2,0,None,0.00,...,NaN,0,0,5,0,0.000000,2024-12-31,9999-12-31,1,France
2,8,15565879,28,Female,108,845,2,0,None,0.00,...,NaN,0,0,5,0,0.000000,2024-12-31,9999-12-31,1,France
3,9,15565891,39,Male,96,709,2,0,None,0.00,...,NaN,0,0,0,0,NaN,2024-12-31,9999-12-31,1,France
4,10,15565996,44,Male,96,653,2,0,None,0.00,...,NaN,0,0,5,1,0.200000,2024-12-31,9999-12-31,1,France
5,13,15566111,39,Male,108,596,1,0,None,0.00,...,NaN,0,0,4,2,0.500000,2024-12-31,9999-12-31,1,France
6,14,15566139,37,Female,60,526,1,0,None,6706.45,...,NaN,0,0,2,0,0.000000,2024-12-31,9999-12-31,1,France
7,17,15566251,37,Female,60,618,1,1,2024-05,11561.51,...,NaN,0,0,0,0,NaN,2024-12-31,9999-12-31,1,France
8,19,15566269,25,Male,60,787,2,0,None,0.00,...,NaN,0,0,2,0,0.000000,2024-12-31,9999-12-31,1,France
9,21,15566295,33,Female,72,761,1,0,None,20083.84,...,NaN,0,0,5,0,0.000000,2024-12-31,9999-12-31,1,France


In [5]:
# read ONE partition folder directly
one = spark.read.parquet("/user/student/churn/warehouse/dim_customer/geography=France")
one.printSchema()      # 20 columns
one.count()           

root
 |-- cust_key: long (nullable = true)
 |-- customer_id: string (nullable = true)
 |-- age: integer (nullable = true)
 |-- gender: string (nullable = true)
 |-- tenure_months: integer (nullable = true)
 |-- credit_score: integer (nullable = true)
 |-- num_products: integer (nullable = true)
 |-- is_churned: integer (nullable = true)
 |-- churn_month: string (nullable = true)
 |-- clv_ltv: double (nullable = true)
 |-- avg_monthly_balance: double (nullable = true)
 |-- avg_ticket_res_time_hrs: double (nullable = true)
 |-- total_tickets: integer (nullable = true)
 |-- high_severity_tickets: integer (nullable = true)
 |-- offers_received: integer (nullable = true)
 |-- offers_accepted: integer (nullable = true)
 |-- offer_acceptance_rate: double (nullable = true)
 |-- dw_start_date: date (nullable = true)
 |-- dw_end_date: date (nullable = true)
 |-- is_current: integer (nullable = true)



5014

# Task 4 - Hive warehousing

In [6]:
spark.sql("SHOW DATABASES").show()

2026-09-03 15:29:46,468 WARN conf.HiveConf: HiveConf of name hive.stats.jdbc.timeout does not exist
2026-09-03 15:29:46,469 WARN conf.HiveConf: HiveConf of name hive.stats.retries.wait does not exist


+---------+
|namespace|
+---------+
| churn_dw|
|  default|
|retail_db|
+---------+



In [7]:
spark.sql("SHOW TABLES IN churn_dw").toPandas()

2026-09-03 15:30:56,941 WARN metastore.ObjectStore: Failed to get database global_temp, returning NoSuchObjectException


,database,tableName,isTemporary
0,churn_dw,dim_customer,False


In [8]:
spark.sql("SHOW PARTITIONS churn_dw.dim_customer").toPandas()

,partition
0,geography=France
1,geography=Germany
2,geography=Spain


In [9]:
spark.sql("""
SELECT COUNT(*) AS rows, SUM(is_churned) AS churned,
       COUNT(DISTINCT customer_id) AS distinct_ids
FROM churn_dw.dim_customer
""").toPandas()

2026-09-03 15:31:33,691 WARN session.SessionState: METASTORE_FILTER_HOOK will be ignored, since hive.security.authorization.manager is set to instance of HiveAuthorizerFactory.
2026-09-03 15:31:33,769 WARN conf.HiveConf: HiveConf of name hive.internal.ss.authz.settings.applied.marker does not exist
2026-09-03 15:31:33,769 WARN conf.HiveConf: HiveConf of name hive.stats.jdbc.timeout does not exist
2026-09-03 15:31:33,769 WARN conf.HiveConf: HiveConf of name hive.stats.retries.wait does not exist


,rows,churned,distinct_ids
0,10000,2037,10000


In [10]:
spark.sql("""
SELECT geography, COUNT(*) AS customers, SUM(is_churned) AS churned
FROM churn_dw.dim_customer GROUP BY geography ORDER BY customers DESC
""").toPandas()

,geography,customers,churned
0,France,5014,810
1,Germany,2509,814
2,Spain,2477,413


In [11]:
spark.sql("""
SELECT customer_id, age, gender, geography, tenure_months, num_products,
       is_churned, churn_month, ROUND(clv_ltv,2) AS clv,
       total_tickets, offers_received
FROM churn_dw.dim_customer
WHERE customer_id IN ('15634602','15647311','15619304')
""").toPandas()

,customer_id,age,gender,geography,tenure_months,num_products,is_churned,churn_month,clv,total_tickets,offers_received
0,15619304,42,Female,France,96,4,1,2024-03,30175.33,1,2
1,15634602,42,Female,France,24,1,1,2024-05,0.00,0,5
2,15647311,41,Female,Spain,12,1,0,None,2000.04,0,4


In [12]:
spark.sql("""
WITH churned_by_month AS (
  SELECT churn_month AS mth, COUNT(*) AS churned_customers
  FROM churn_dw.dim_customer
  WHERE is_current=1 AND is_churned=1 AND churn_month IS NOT NULL
  GROUP BY churn_month),
totals AS (
  SELECT COUNT(*) AS total_customers FROM churn_dw.dim_customer WHERE is_current=1),
cumulative AS (
  SELECT mth, churned_customers,
    COALESCE(SUM(churned_customers) OVER (
      ORDER BY mth ROWS BETWEEN UNBOUNDED PRECEDING AND 1 PRECEDING),0) AS churned_before
  FROM churned_by_month)
SELECT mth AS churn_month, churned_customers,
       total_customers - churned_before AS active_at_start,
       ROUND(100.0*churned_customers/(total_customers-churned_before),2) AS churn_rate_pct
FROM cumulative CROSS JOIN totals ORDER BY mth
""").toPandas()

2026-09-03 15:33:14,257 WARN window.WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
2026-09-03 15:33:14,391 WARN conf.HiveConf: HiveConf of name hive.internal.ss.authz.settings.applied.marker does not exist
2026-09-03 15:33:14,391 WARN conf.HiveConf: HiveConf of name hive.stats.jdbc.timeout does not exist
2026-09-03 15:33:14,392 WARN conf.HiveConf: HiveConf of name hive.stats.retries.wait does not exist


,churn_month,churned_customers,active_at_start,churn_rate_pct
0,2024-01,238,10000,2.38
1,2024-02,235,9762,2.41
2,2024-03,223,9527,2.34
3,2024-04,218,9304,2.34
4,2024-05,219,9086,2.41
5,2024-06,244,8867,2.75
6,2024-07,228,8623,2.64
7,2024-08,225,8395,2.68
8,2024-09,207,8170,2.53
